# 5. Human-in-the-Loop / Interrupts

`interrupt()` pauses a graph mid-node and surfaces a value to the caller; the
caller resumes execution later with `Command(resume=...)`. This is a distinctly
LangGraph feature — a plain LangChain chain either finishes or it doesn't, but a
graph can pause indefinitely awaiting a human decision. Interrupts require a
checkpointer, since the paused state has to be persisted between the pausing
call and the resuming call.

Demonstrated with a draft → request_approval → finalize graph.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from models.chat_models.ollama_models import SupportedModel, get_chat_model


class ApprovalState(TypedDict):
    request: str
    draft: str
    approved: bool
    result: str


llm = get_chat_model(SupportedModel.llama3_2)


def draft(state: ApprovalState) -> dict:
    response = llm.invoke(f"Draft a short (1-2 sentence) reply to: {state['request']}")
    return {"draft": response.content}


def request_approval(state: ApprovalState) -> dict:
    decision = interrupt({"question": "Approve this draft?", "draft": state["draft"]})
    return {"approved": bool(decision)}


def finalize(state: ApprovalState) -> dict:
    if state["approved"]:
        return {"result": f"[SENT] {state['draft']}"}
    return {"result": "[REJECTED] Draft was not sent."}


graph = StateGraph(ApprovalState)
graph.add_node("draft", draft)
graph.add_node("request_approval", request_approval)
graph.add_node("finalize", finalize)
graph.add_edge(START, "draft")
graph.add_edge("draft", "request_approval")
graph.add_edge("request_approval", "finalize")
graph.add_edge("finalize", END)
compiled = graph.compile(checkpointer=InMemorySaver())

## Start — drafts a reply, then pauses

In [ ]:
thread = {"configurable": {"thread_id": "notebook-demo"}}
started = compiled.invoke(
    {"request": "Can we reschedule tomorrow's meeting?", "draft": "", "approved": False, "result": ""}, thread
)

pending = started["__interrupt__"][0].value
print("Waiting for approval on draft:")
print(pending["draft"])

## Resume — approve it

In [ ]:
resumed = compiled.invoke(Command(resume=True), thread)
print(resumed["result"])

## 🧪 Playground

**1. Reject instead** — start a *new* thread (`thread_id="notebook-demo-2"`), then resume with `Command(resume=False)`.

In [ ]:
# TODO: start a new thread, then resume with resume=False


**2. Resume with no pending interrupt** — call `compiled.invoke(Command(resume=True), thread)` a *second* time on the already-finished `"notebook-demo"` thread and see what error LangGraph raises. (The app wraps this in a friendlier `ValueError` — see `langgraph_demo/interrupts.py`'s `resume_approval_demo`.)

In [ ]:
# TODO: try resuming an already-finished thread


**3. Check pending state directly** — `compiled.get_state(thread).next` tells you whether a thread is currently paused, without needing to resume it.

In [ ]:
# TODO: compiled.get_state(thread).next on a fresh, started-but-not-resumed thread
